In [1]:
import sys
import os
from pathlib import Path

# Thêm thư mục cha (rag-service) vào danh sách tìm kiếm của Python
notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)

# Tạo Dữ liệu Benchmark với RAGAS

Notebook này dùng để sinh test set cho đánh giá RAG system sử dụng RAGAS framework.

## RAGAS trả về gì?

Sau khi chạy RAGAS TestsetGenerator, nó trả về một **test set** chứa:
- **question**: Câu hỏi được sinh tự động từ documents
- **ground_truth**: Câu trả lời đúng (được LLM critic đánh giá)
- **context**: Các documents liên quan được retrieve
- **synthesizer_type**: Loại câu hỏi (simple, reasoning, multi_context)

## Lưu trữ ở đâu?

Test set được lưu vào file JSONL tại:
```
src/h_evaluation/test_sets/ragas_generated.jsonl
```

## Cách dùng để đánh giá benchmark?

**RAGAS đánh giá phụ thuộc vào test set đã có sẵn** - nó KHÔNG sinh câu hỏi mới khi đánh giá. Quy trình:

1. **Tạo test set** (notebook này) → lưu vào file JSONL
2. **Chạy RAG system** với các câu hỏi trong test set → thu thập answers
3. **Đánh giá với RAGAS metrics**:
   - Faithfulness: Answer có trung thực với context không?
   - Answer Relevancy: Answer có liên quan đến question không?
   - Context Precision: Context có chứa thông tin cần thiết không?
   - Context Recall: Context có đủ thông tin để trả lời không?

In [2]:
# Import các module cần thiết
from src.h_evaluation.generate_benchmark_data import load_documents, setup_generator, generate_testset, spot_check_testset

c:\Users\Admin\anaconda3\envs\DL\Lib\site-packages\instructor\providers\gemini\client.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 1. Load Documents từ ChromaDB

Load documents từ 2 nguồn:
- `products_collection`: Mô tả sản phẩm
- `policies_collection`: Chính sách shop

In [3]:
# Load documents (có thể chỉnh limit_products, limit_policies)
documents = load_documents(limit_products=100, limit_policies=50)
print(f"📚 Total documents loaded: {len(documents)}")

📦 Loading products from ChromaDB...
   ✅ Loaded 100 products
📜 Loading policies from ChromaDB...
   ✅ Loaded 3 policies
📚 Total documents loaded: 103
📚 Total documents loaded: 103


## 2. Setup RAGAS Generator

Dùng Groq (LLM) + OpenRouter (Embedding)
- Groq: 30 req/min, 1,000 req/day
- OpenRouter: Embedding model từ config

In [4]:
# Setup generator (Groq + OpenRouter)
generator = setup_generator()
print("✅ Generator setup with Groq (LLM) + OpenRouter (Embedding)")

⚙️  SETUP GENERATOR
🤖 LLM Model: openai/gpt-oss-120b
🔑 Groq API Key: gsk_mderM6...
🔧 Setting up Groq client via OpenAI API...
✅ Groq LLM setup complete
🔤 Embedding Model: nvidia/llama-nemotron-embed-vl-1b-v2:free
🔑 OpenRouter API Key: sk-or-v1-e...
🔧 Setting up OpenRouter client...
✅ OpenRouter Embedding setup complete
✅ TestsetGenerator created
✅ Generator setup with Groq (LLM) + OpenRouter (Embedding)


In [ ]:
# Generate test set
num_samples = 5  # Số lượng câu hỏi sinh ra
output_path = "src/h_evaluation/test_sets/ragas_generated.jsonl"

testset = generate_testset(
    documents=documents,
    output_path=output_path,
    num_samples=num_samples
)

# Spot-check test set
spot_check_testset(output_path, num_check=5)

🚀 BẮT ĐẦU SINH TEST SET
📚 Số lượng documents đầu vào: 103
🎯 Số lượng câu hỏi cần sinh: 5
⚙️  SETUP GENERATOR
🤖 LLM Model: openai/gpt-oss-120b
🔑 Groq API Key: gsk_mderM6...
🔧 Setting up Groq client via OpenAI API...
✅ Groq LLM setup complete
🔤 Embedding Model: nvidia/llama-nemotron-embed-vl-1b-v2:free
🔑 OpenRouter API Key: sk-or-v1-e...
🔧 Setting up OpenRouter client...
✅ OpenRouter Embedding setup complete
✅ TestsetGenerator created
✅ Generator setup hoàn tất
⚙️  RunConfig: max_workers=1 (sequential), max_retries=5

🔄 BẮT ĐẦU GỌI RAGAS...
   - Bước 1: Embedding documents (OpenRouter)
   - Bước 2: Apply transforms (HeadlinesExtractor, Splitter, etc.)
   - Bước 3: Generate questions (Groq)
------------------------------------------------------------


Applying HeadlinesExtractor:   0%|          | 0/90 [00:00<?, ?it/s]